# Подготовка локального zip-датасета подкатегорий

Этот ноутбук является локальной серверной версией `01_prepare_drive_subcategory_dataset.ipynb`. Он берет zip-архив из папки проекта `data`, распаковывает его, сканирует изображения и сохраняет те же артефакты `wardrobe_classifier_outputs`, которые ожидает ноутбук обучения.

Поддерживаемая структура внутри архива:

- `category/subcategory/image.jpg`
- `subcategory/image.jpg`
- `train/category/subcategory/image.jpg`, `val/...`, `test/...`
- `train/subcategory/image.jpg`, `val/...`, `test/...`

Если в `data` лежит несколько zip-архивов, укажите нужный файл в переменной `ZIP_PATH`.


## Установка и проверка окружения


In [ ]:
# Если зависимости уже установлены на сервере, эту ячейку можно не запускать.
%pip -q install pandas numpy pillow tqdm

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

RANDOM_SEED = 42
# Ноутбук обучения читает файлы через tf.image.decode_jpeg,
# поэтому здесь оставлены только JPEG-расширения.
IMAGE_EXTENSIONS = {".jpg", ".jpeg"}
SPLIT_NAMES = ("train", "val", "test")

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "wardrobe_classifier_outputs"

# Если в data несколько архивов, раскомментируйте и укажите нужный файл:
# ZIP_PATH = DATA_DIR / "dataset.zip"
ZIP_PATH = None

FORCE_EXTRACT = False
CHECK_IMAGE_READABILITY = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Папка проекта:", PROJECT_DIR)
print("Папка data:", DATA_DIR)
print("Папка артефактов:", OUTPUT_DIR)
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Не найдена папка data: {DATA_DIR}")


## Поиск и распаковка архива


In [ ]:
def find_zip_path(data_dir: Path, zip_path=None) -> Path:
    if zip_path is not None:
        zip_path = Path(zip_path)
        if not zip_path.exists():
            raise FileNotFoundError(f"Указанный zip-архив не найден: {zip_path}")
        return zip_path

    archives = sorted(path for path in data_dir.glob("*.zip") if path.is_file())
    if not archives:
        raise FileNotFoundError(f"В папке {data_dir} не найден zip-архив. Положите датасет в data/*.zip.")
    if len(archives) > 1:
        archive_list = "\n".join(f"- {path.name}" for path in archives)
        raise RuntimeError("В папке data найдено несколько zip-архивов. Укажите нужный в ZIP_PATH:\n" + archive_list)
    return archives[0]


def safe_extract_zip(zip_path: Path, target_dir: Path, force: bool = False) -> Path:
    target_dir.mkdir(parents=True, exist_ok=True)
    if any(target_dir.iterdir()) and not force:
        print("Архив уже распакован, используется существующая папка:", target_dir)
        return target_dir

    with zipfile.ZipFile(zip_path) as archive:
        target_root = target_dir.resolve()
        for member in archive.infolist():
            destination = (target_dir / member.filename).resolve()
            if destination != target_root and target_root not in destination.parents:
                raise RuntimeError(f"Небезопасный путь внутри архива: {member.filename}")
        archive.extractall(target_dir)
    print("Архив распакован в:", target_dir)
    return target_dir


def meaningful_dirs(path: Path):
    return [
        child for child in sorted(path.iterdir())
        if child.is_dir() and not child.name.startswith(".") and child.name != "__MACOSX"
    ]


def has_images_directly(path: Path) -> bool:
    return any(child.is_file() and child.suffix.lower() in IMAGE_EXTENSIONS for child in path.iterdir())


def unwrap_single_root(path: Path) -> Path:
    root = path
    while True:
        children = meaningful_dirs(root)
        if len(children) == 1 and not has_images_directly(root):
            root = children[0]
            continue
        return root

ZIP_PATH = find_zip_path(DATA_DIR, ZIP_PATH)
EXTRACT_DIR = DATA_DIR / f"_{ZIP_PATH.stem}_extracted"
DATASET_DIR = unwrap_single_root(safe_extract_zip(ZIP_PATH, EXTRACT_DIR, force=FORCE_EXTRACT))

print("Zip-архив:", ZIP_PATH)
print("Папка датасета:", DATASET_DIR)
if not DATASET_DIR.exists():
    raise FileNotFoundError(f"Не найдена папка датасета после распаковки: {DATASET_DIR}")


## Сканирование изображений


In [ ]:
def is_image(path: Path) -> bool:
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def clean_parts(parts):
    return [part for part in parts if part and not part.startswith(".") and part != "__MACOSX"]


def labels_from_relative_parent(relative_parent: Path):
    """Определяет category/subcategory из структуры папок без ручного маппинга.

    Поддерживаются варианты:
    - Датасет_подкатегории/category/subcategory/image.jpg
    - Датасет_подкатегории/subcategory/image.jpg
    - Датасет_подкатегории/train/category/subcategory/image.jpg
    - Датасет_подкатегории/train/subcategory/image.jpg
    """
    parts = clean_parts(relative_parent.parts)
    if not parts:
        raise ValueError("Изображения должны лежать внутри папок классов, а не в корне датасета.")
    if len(parts) == 1:
        subcategory = parts[0]
        category = parts[0]
    else:
        category = parts[0]
        subcategory = parts[-1]
    return category, subcategory


def detect_existing_split(dataset_dir: Path) -> bool:
    return any((dataset_dir / split).is_dir() for split in SPLIT_NAMES)


def scan_images_with_existing_split(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for split in SPLIT_NAMES:
        split_dir = dataset_dir / split
        if not split_dir.is_dir():
            continue
        for image_path in sorted(split_dir.rglob("*")):
            if not is_image(image_path):
                continue
            relative_parent = image_path.parent.relative_to(split_dir)
            category, subcategory = labels_from_relative_parent(relative_parent)
            rows.append({
                "article_id": image_path.stem,
                "image_path": str(image_path),
                "source_product_type_name": "",
                "source_product_group_name": "",
                "target_category": category,
                "target_subcategory": subcategory,
                "mapping_rule": "folder_structure",
                "split": split,
            })
    return pd.DataFrame(rows)


def scan_images_without_split(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for image_path in sorted(dataset_dir.rglob("*")):
        if not is_image(image_path):
            continue
        relative_parent = image_path.parent.relative_to(dataset_dir)
        category, subcategory = labels_from_relative_parent(relative_parent)
        rows.append({
            "article_id": image_path.stem,
            "image_path": str(image_path),
            "source_product_type_name": "",
            "source_product_group_name": "",
            "target_category": category,
            "target_subcategory": subcategory,
            "mapping_rule": "folder_structure",
        })
    return pd.DataFrame(rows)


has_existing_split = detect_existing_split(DATASET_DIR)
samples = scan_images_with_existing_split(DATASET_DIR) if has_existing_split else scan_images_without_split(DATASET_DIR)

if samples.empty:
    raise RuntimeError(f"В папке {DATASET_DIR} не найдены изображения с расширениями: {sorted(IMAGE_EXTENSIONS)}")

samples["article_id"] = samples["article_id"].astype("string")
samples = samples.drop_duplicates(subset=["image_path"]).reset_index(drop=True)

print("Найден готовый split train/val/test:" if has_existing_split else "Готовый split не найден, он будет создан ниже.")
print("Всего изображений:", len(samples))
print("Категорий:", samples["target_category"].nunique())
print("Подкатегорий:", samples["target_subcategory"].nunique())
display(samples.head())
display(samples.groupby(["target_category", "target_subcategory"], as_index=False).size().rename(columns={"size": "image_count"}).sort_values("image_count"))

In [ ]:
def split_sizes_for_count(count: int):
    """Правила split сохранены из исходного прототипа."""
    if count >= 300:
        val = round(0.1 * count)
        test = round(0.1 * count)
    elif count >= 100:
        val = max(10, round(0.1 * count))
        test = max(10, round(0.1 * count))
    else:
        val = min(10, max(6, round(0.12 * count)))
        test = min(10, max(6, round(0.12 * count)))

    if count >= 3 and val + test >= count:
        val = max(1, min(val, (count - 1) // 2))
        test = max(1, min(test, count - val - 1))
    elif count == 2:
        val, test = 0, 1
    elif count == 1:
        val, test = 0, 0
    return int(count - val - test), int(val), int(test)


def make_stratified_split(df: pd.DataFrame, seed: int = RANDOM_SEED):
    rng = np.random.default_rng(seed)
    parts, report_rows = [], []
    for subcategory, group in df.groupby("target_subcategory", sort=False):
        group = group.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        count = len(group)
        train_n, val_n, test_n = split_sizes_for_count(count)
        labels = np.array(["train"] * count, dtype=object)
        labels[train_n:train_n + val_n] = "val"
        labels[train_n + val_n:train_n + val_n + test_n] = "test"
        group = group.iloc[rng.permutation(count)].copy()
        group["split"] = labels
        parts.append(group)
        status = "rare" if count < 100 else "ok"
        if count < 3:
            status = "low_image_count"
        report_rows.append({"target_subcategory": subcategory, "count": count, "train": train_n, "val": val_n, "test": test_n, "status": status})
    return pd.concat(parts, ignore_index=True), pd.DataFrame(report_rows).sort_values("count")


def build_split_report(df: pd.DataFrame) -> pd.DataFrame:
    counts = df.groupby(["target_subcategory", "split"], as_index=False).size()
    table = counts.pivot(index="target_subcategory", columns="split", values="size").fillna(0).astype(int)
    for split in SPLIT_NAMES:
        if split not in table.columns:
            table[split] = 0
    table["count"] = table[list(SPLIT_NAMES)].sum(axis=1)
    table["status"] = np.where(table["count"] < 3, "low_image_count", np.where(table["count"] < 100, "rare", "ok"))
    return table.reset_index()[["target_subcategory", "count", "train", "val", "test", "status"]].sort_values("count")


if has_existing_split:
    split_report = build_split_report(samples)
else:
    samples, split_report = make_stratified_split(samples)

display(split_report)
display(pd.crosstab(samples["target_subcategory"], samples["split"]).sort_values("train"))

In [ ]:
def filter_readable_images(df: pd.DataFrame) -> pd.DataFrame:
    if not CHECK_IMAGE_READABILITY:
        exists_mask = df["image_path"].map(lambda value: Path(value).exists())
        missing = int((~exists_mask).sum())
        if missing:
            print("Отсутствующих файлов:", missing)
        return df[exists_mask].copy()

    keep = []
    for path in tqdm(df["image_path"], desc="Проверка изображений"):
        try:
            with Image.open(path) as image:
                image.verify()
            keep.append(True)
        except Exception:
            keep.append(False)
    keep = np.array(keep, dtype=bool)
    print("Нечитаемых изображений:", int((~keep).sum()))
    return df[keep].copy()


samples = filter_readable_images(samples).reset_index(drop=True)
if samples.empty:
    raise RuntimeError("После проверки файлов не осталось изображений для обучения.")

taxonomy_df = (
    samples[["target_category", "target_subcategory"]]
    .drop_duplicates()
    .sort_values(["target_category", "target_subcategory"])
    .reset_index(drop=True)
)
category_names = taxonomy_df["target_category"].drop_duplicates().tolist()
subcategory_names = taxonomy_df["target_subcategory"].tolist()
category_to_id = {name: idx for idx, name in enumerate(category_names)}
subcategory_to_id = {name: idx for idx, name in enumerate(subcategory_names)}
subcategory_to_category = dict(zip(taxonomy_df["target_subcategory"], taxonomy_df["target_category"]))

category_counts = samples["target_category"].value_counts().rename_axis("target_category").reset_index(name="image_count")
subcategory_counts = samples["target_subcategory"].value_counts().rename_axis("target_subcategory").reset_index(name="image_count")
category_subcategory_counts = (
    samples.groupby(["target_category", "target_subcategory"], as_index=False)
    .size().rename(columns={"size": "image_count"})
    .sort_values(["target_category", "image_count"], ascending=[True, False])
)

display(taxonomy_df)
display(category_counts)
display(subcategory_counts)
display(category_subcategory_counts)

## Сохранение CSV/JSON артефактов


In [ ]:
columns = [
    "article_id", "image_path", "source_product_type_name", "source_product_group_name",
    "target_category", "target_subcategory", "mapping_rule", "split",
]
samples = samples[columns].copy()

taxonomy_df.to_csv(OUTPUT_DIR / "taxonomy.csv", index=False)
samples.to_csv(OUTPUT_DIR / "samples.csv", index=False)
samples[samples["split"] == "train"].to_csv(OUTPUT_DIR / "train.csv", index=False)
samples[samples["split"] == "val"].to_csv(OUTPUT_DIR / "val.csv", index=False)
samples[samples["split"] == "test"].to_csv(OUTPUT_DIR / "test.csv", index=False)
split_report.to_csv(OUTPUT_DIR / "split_report.csv", index=False)
category_counts.to_csv(OUTPUT_DIR / "category_counts.csv", index=False)
subcategory_counts.to_csv(OUTPUT_DIR / "subcategory_counts.csv", index=False)
category_subcategory_counts.to_csv(OUTPUT_DIR / "category_subcategory_counts.csv", index=False)

for name, obj in {
    "category_to_id.json": category_to_id,
    "subcategory_to_id.json": subcategory_to_id,
    "subcategory_to_category.json": subcategory_to_category,
    "category_names.json": category_names,
    "subcategory_names.json": subcategory_names,
}.items():
    (OUTPUT_DIR / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

print("Сохранены артефакты:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path)

print("\nИтог:")
print("Всего изображений:", len(samples))
print("Категорий:", len(category_names))
print("Подкатегорий:", len(subcategory_names))
print("Train / val / test:")
print(samples["split"].value_counts().to_string())

## Дополнительно: локальное копирование


In [ ]:
def copy_outputs_to_dir(target_dir):
    """Копирует CSV/JSON артефакты в указанную локальную папку."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            shutil.copy2(path, target_dir / path.name)
    print("Артефакты скопированы в:", target_dir)
    return target_dir


# Пример:
# copy_outputs_to_dir(PROJECT_DIR / "some_other_outputs_dir")
